# BimpeTTS — Install → Load → Generate

End-to-end notebook for **finetuned** BimpeTTS (LibriTTS base + HiFi-GAN, reference-conditioned).

| Step | What |
|---|---|
| 1 | Install / repair Python deps |
| 2 | Configure paths + verify checkpoint & reference wav |
| 3 | Load StyleTTS2 model |
| 4 | Generate speech from text |

**Before this notebook:** finetune must already be done (`python train_finetune.py --config_path Configs/config_bimpe_ft.yml`).

Run cells **top to bottom**. After install cells, **restart the kernel**, then continue from **Configure paths**.

## 1) Install / setup

Run once on a fresh machine / container. Skip if deps are already working.

After installs: **Kernel → Restart**, then start again at section 2.

In [ ]:
# Move to StyleTTS2 repo root (notebook lives in Demo/)
%cd ..
!pwd
!ls Configs/config_bimpe_ft.yml Models train_finetune.py 2>/dev/null || true

In [ ]:
# System packages for phonemizer (Debian/Ubuntu). May need sudo outside Docker.
import shutil, subprocess, sys

if shutil.which('apt-get'):
    subprocess.run(['apt-get', 'update', '-qq'], check=False)
    subprocess.run(
        ['apt-get', 'install', '-y', '-qq', 'espeak-ng', 'espeak-ng-data', 'libespeak-ng1', 'build-essential'],
        check=False,
    )
else:
    print('apt-get not found — install espeak-ng manually if phonemizer fails')

In [ ]:
# Python deps (pinned for torch 2.4.x + StyleTTS2)
# IMPORTANT: do NOT blank-upgrade transformers or numpy.
import sys
!{sys.executable} -m pip install -q --upgrade pip
!{sys.executable} -m pip install -q \
  "numpy==1.26.4" \
  "scipy==1.11.4" \
  "transformers==4.51.3" \
  "librosa==0.10.1" \
  "SoundFile" "munch" "pydub" "pyyaml" "nltk" "matplotlib" \
  "accelerate" "einops" "einops-exts" "tqdm" "typing-extensions" \
  "phonemizer" "ipython" "torchaudio"

# Resemble AI monotonic_align (has mask_from_lens)
!{sys.executable} -m pip uninstall -y monotonic-align monotonic_align >/dev/null 2>&1 || true
!{sys.executable} -m pip install -q --no-cache-dir git+https://github.com/resemble-ai/monotonic_align.git

In [ ]:
# NLTK tokenizer data
import nltk
for pkg in ['punkt', 'punkt_tab']:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(pkg, e)
print('nltk ready')

In [ ]:
# Verify critical imports
import torch, numpy, transformers, librosa, phonemizer
from monotonic_align.core import maximum_path_c
try:
    from monotonic_align import mask_from_lens
    print('mask_from_lens: package OK')
except ImportError:
    print('mask_from_lens: missing in package (utils.py local fallback is OK)')

print('python :', __import__('sys').executable)
print('torch  :', torch.__version__, '| cuda:', torch.cuda.is_available())
print('numpy  :', numpy.__version__)
print('transf :', transformers.__version__)
print('librosa:', librosa.__version__)
print('\n>>> Restart kernel now, then continue from section 2 <<<')

## 2) Configure paths

Edit these if your server layout differs.

In [ ]:
import os, glob, sys
from pathlib import Path

# Ensure we're at repo root
if Path('Demo').exists() and Path('Configs').exists():
    REPO_ROOT = Path('.').resolve()
elif Path('../Configs').exists():
    %cd ..
    REPO_ROOT = Path('.').resolve()
else:
    raise RuntimeError('Run from StyleTTS2 repo (or Demo/).')

print('REPO_ROOT =', REPO_ROOT)

# ---- EDIT THESE ----
CONFIG_PATH = 'Configs/config_bimpe_ft.yml'
CHECKPOINT_PATH = None  # None => latest under Models/BimpeTTS_ft/epoch_2nd_*.pth
REF_WAV = '/root/BimpeTTS_Dataset/wavs/bimpe_0001.wav'
WAVS_DIR = '/root/BimpeTTS_Dataset/wavs'
OUT_DIR = 'Demo/bimpe_ft_outputs'
SR = 24000
# --------------------

os.makedirs(OUT_DIR, exist_ok=True)

def find_latest_checkpoint(log_dir='Models/BimpeTTS_ft'):
    paths = sorted(glob.glob(os.path.join(log_dir, 'epoch_2nd_*.pth')))
    if not paths:
        raise FileNotFoundError(
            f'No checkpoint in {log_dir}. Finetune first:\n'
            f'  python train_finetune.py --config_path Configs/config_bimpe_ft.yml'
        )
    return paths[-1]

ckpt = CHECKPOINT_PATH or find_latest_checkpoint()
print('CONFIG     :', CONFIG_PATH, 'exists=' + str(os.path.isfile(CONFIG_PATH)))
print('CHECKPOINT :', ckpt, 'exists=' + str(os.path.isfile(ckpt)))
print('REF_WAV    :', REF_WAV, 'exists=' + str(os.path.isfile(REF_WAV)))
print('WAVS_DIR   :', WAVS_DIR, 'exists=' + str(os.path.isdir(WAVS_DIR)))

assert os.path.isfile(CONFIG_PATH), f'Missing config: {CONFIG_PATH}'
assert os.path.isfile(ckpt), f'Missing checkpoint: {ckpt}'
assert os.path.isfile(REF_WAV), f'Missing reference wav: {REF_WAV} — pick another under {WAVS_DIR}'
CHECKPOINT_PATH = ckpt
print('\nPaths OK')

## 3) Imports + helpers

In [ ]:
import torch
torch.manual_seed(0)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

import random
random.seed(0)
import numpy as np
np.random.seed(0)

import re
import time
import yaml
import torchaudio
import librosa
from nltk.tokenize import word_tokenize
import IPython.display as ipd

from models import *
from utils import *
from text_utils import TextCleaner
from Modules.diffusion.sampler import DiffusionSampler, ADPM2Sampler, KarrasSchedule

textcleaner = TextCleaner()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

to_mel = torchaudio.transforms.MelSpectrogram(
    n_mels=80, n_fft=2048, win_length=1200, hop_length=300)
mean, std = -4, 4

def length_to_mask(lengths):
    mask = torch.arange(lengths.max()).unsqueeze(0).expand(lengths.shape[0], -1).type_as(lengths)
    return torch.gt(mask + 1, lengths.unsqueeze(1))

def preprocess(wave):
    wave_tensor = torch.from_numpy(wave).float()
    mel_tensor = to_mel(wave_tensor)
    return (torch.log(1e-5 + mel_tensor.unsqueeze(0)) - mean) / std

def save_wav(path, audio, sr=SR):
    audio = np.asarray(audio, dtype=np.float32)
    peak = np.max(np.abs(audio)) + 1e-8
    if peak > 1.0:
        audio = audio / peak
    torchaudio.save(path, torch.from_numpy(audio).unsqueeze(0), sr)
    print('saved:', path)

def compute_style(path):
    wave, sr = librosa.load(path, sr=24000)
    audio, _ = librosa.effects.trim(wave, top_db=30)
    if sr != 24000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=24000)
    mel_tensor = preprocess(audio).to(device)
    with torch.no_grad():
        ref_s = model.style_encoder(mel_tensor.unsqueeze(1))
        ref_p = model.predictor_encoder(mel_tensor.unsqueeze(1))
    return torch.cat([ref_s, ref_p], dim=1)

## 4) Load model

In [ ]:
import phonemizer
global_phonemizer = phonemizer.backend.EspeakBackend(
    language='en-us', preserve_punctuation=True, with_stress=True)
print('phonemizer OK')

In [ ]:
from pathlib import Path

cfg_path = Path(CONFIG_PATH).resolve()
print('loading config:', cfg_path)
print('size bytes:', cfg_path.stat().st_size)

# Guard against accidentally pointing CONFIG_PATH at a .pth checkpoint
raw = cfg_path.read_bytes()[:500]
if b'log_dir' not in raw:
    raise ValueError(
        f'CONFIG_PATH does not look like a YAML config file: {cfg_path}\n'
        f'head={raw[:64]!r}\n'
        f'Set CONFIG_PATH = "Configs/config_bimpe_ft.yml"'
    )

with open(cfg_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

print('log_dir   :', config.get('log_dir'))
print('multispeaker:', config['model_params'].get('multispeaker'))
print('decoder   :', config['model_params']['decoder']['type'])

text_aligner = load_ASR_models(config.get('ASR_path'), config.get('ASR_config'))
pitch_extractor = load_F0_models(config.get('F0_path'))

from Utils.PLBERT.util import load_plbert
plbert = load_plbert(config.get('PLBERT_dir'))

model_params = recursive_munch(config['model_params'])
model = build_model(model_params, text_aligner, pitch_extractor, plbert)
_ = [model[key].eval() for key in model]
_ = [model[key].to(device) for key in model]
print('architecture built')


In [ ]:
from collections import OrderedDict

print('loading checkpoint:', CHECKPOINT_PATH)
params_whole = torch.load(CHECKPOINT_PATH, map_location='cpu')
params = params_whole['net']
print('ckpt epoch:', params_whole.get('epoch'), '| val_loss:', params_whole.get('val_loss'))

for key in model:
    if key not in params:
        continue
    print('%s loaded' % key)
    state_dict = params[key]
    try:
        model[key].load_state_dict(state_dict, strict=True)
    except Exception:
        new_state_dict = OrderedDict()
        if next(iter(state_dict)).startswith('module.'):
            for k, v in state_dict.items():
                new_state_dict[k[7:] if k.startswith('module.') else k] = v
        else:
            new_state_dict = state_dict
        missing, unexpected = model[key].load_state_dict(new_state_dict, strict=False)
        if missing:
            print('  missing', len(missing))
        if unexpected:
            print('  unexpected', len(unexpected))

_ = [model[key].eval() for key in model]

sampler = DiffusionSampler(
    model.diffusion.diffusion,
    sampler=ADPM2Sampler(),
    sigma_schedule=KarrasSchedule(sigma_min=0.0001, sigma_max=3.0, rho=9.0),
    clamp=False,
)
print('model ready')

## 5) Reference voice + text to speak

In [ ]:
ref_audio, _ = librosa.load(REF_WAV, sr=SR)
print('Reference voice:', REF_WAV)
display(ipd.Audio(ref_audio, rate=SR))

ref_s = compute_style(REF_WAV)
print('ref_s shape:', tuple(ref_s.shape))

In [ ]:
# Paste any length paragraph — it will be split by sentence automatically.
text = '''
Hello, my name is Bimpe. Welcome to this text to speech demo.
I soon realized that building something meaningful takes patience and consistency.
Getting there was an adventure, and I could not be more grateful for the lessons.
'''
print(text.strip())
print('--- chars:', len(text.strip()))

## 6) Synthesis functions

In [ ]:
def split_sentences(text):
    text = text.strip().replace('"', '')
    parts = re.split(r'(?<=[.!?])\s+', text)
    out = []
    for p in parts:
        p = p.strip()
        if not p:
            continue
        if p[-1] not in '.!?':
            p += '.'
        out.append(p)
    return out


def synthesize_sentence(text, ref_s, s_prev=None, alpha=0.3, beta=0.7, t=0.7,
                        diffusion_steps=5, embedding_scale=1, speed=1.0, max_tokens=500):
    text = text.strip().replace('"', '')
    ps = global_phonemizer.phonemize([text])
    ps = word_tokenize(ps[0])
    ps = ' '.join(ps).replace('``', '"').replace("''", '"')

    tokens = textcleaner(ps)
    tokens.insert(0, 0)
    if len(tokens) > max_tokens:
        raise ValueError(f'Sentence too long ({len(tokens)} tokens): {text[:80]}...')
    if speed <= 0:
        raise ValueError(f'speed must be > 0, got {speed}')

    tokens = torch.LongTensor(tokens).to(device).unsqueeze(0)

    with torch.no_grad():
        input_lengths = torch.LongTensor([tokens.shape[-1]]).to(device)
        text_mask = length_to_mask(input_lengths).to(device)

        t_en = model.text_encoder(tokens, input_lengths, text_mask)
        bert_dur = model.bert(tokens, attention_mask=(~text_mask).int())
        d_en = model.bert_encoder(bert_dur).transpose(-1, -2)

        s_pred = sampler(
            noise=torch.randn((1, 256)).unsqueeze(1).to(device),
            embedding=bert_dur,
            embedding_scale=embedding_scale,
            features=ref_s,
            num_steps=diffusion_steps,
        ).squeeze(1)

        if s_prev is not None:
            s_pred = t * s_prev + (1 - t) * s_pred

        s = s_pred[:, 128:]
        ref = s_pred[:, :128]
        ref = alpha * ref + (1 - alpha) * ref_s[:, :128]
        s = beta * s + (1 - beta) * ref_s[:, 128:]
        s_pred = torch.cat([ref, s], dim=-1)

        d = model.predictor.text_encoder(d_en, s, input_lengths, text_mask)
        x, _ = model.predictor.lstm(d)
        duration = torch.sigmoid(model.predictor.duration_proj(x)).sum(axis=-1)
        pred_dur = torch.round(duration.squeeze() / speed).clamp(min=1)

        pred_aln_trg = torch.zeros(input_lengths, int(pred_dur.sum().data))
        c_frame = 0
        for i in range(pred_aln_trg.size(0)):
            pred_aln_trg[i, c_frame:c_frame + int(pred_dur[i].data)] = 1
            c_frame += int(pred_dur[i].data)

        en = (d.transpose(-1, -2) @ pred_aln_trg.unsqueeze(0).to(device))
        if model_params.decoder.type == 'hifigan':
            asr_new = torch.zeros_like(en)
            asr_new[:, :, 0] = en[:, :, 0]
            asr_new[:, :, 1:] = en[:, :, 0:-1]
            en = asr_new

        F0_pred, N_pred = model.predictor.F0Ntrain(en, s)

        asr = (t_en @ pred_aln_trg.unsqueeze(0).to(device))
        if model_params.decoder.type == 'hifigan':
            asr_new = torch.zeros_like(asr)
            asr_new[:, :, 0] = asr[:, :, 0]
            asr_new[:, :, 1:] = asr[:, :, 0:-1]
            asr = asr_new

        out = model.decoder(asr, F0_pred, N_pred, ref.squeeze().unsqueeze(0))

    return out.squeeze().cpu().numpy()[..., :-50], s_pred


def inference(text, ref_s, alpha=0.3, beta=0.7, diffusion_steps=5,
              embedding_scale=1, t=0.7, pause_ms=120, speed=1.0):
    sentences = split_sentences(text)
    if not sentences:
        raise ValueError('No sentences found')

    print(f'Split into {len(sentences)} sentence(s)')
    for i, s in enumerate(sentences):
        print(f'  [{i+1}] {s}')

    wavs, s_prev = [], None
    silence = np.zeros(int(SR * pause_ms / 1000.0), dtype=np.float32)
    for i, sent in enumerate(sentences):
        wav, s_prev = synthesize_sentence(
            sent, ref_s, s_prev=s_prev,
            alpha=alpha, beta=beta, t=t,
            diffusion_steps=diffusion_steps,
            embedding_scale=embedding_scale,
            speed=speed,
        )
        wavs.append(wav.astype(np.float32))
        if i < len(sentences) - 1:
            wavs.append(silence)
    return np.concatenate(wavs)

print('synthesis helpers ready')


## 7) Generate voice

In [ ]:
start = time.time()
wav = inference(
    text, ref_s,
    alpha=0.3, beta=0.7,
    diffusion_steps=10,
    embedding_scale=1,
    pause_ms=120,
    speed=1.0,  # >1 faster, <1 slower (scales pred_dur)
)
elapsed = time.time() - start
print(f'done in {elapsed:.1f}s | audio {len(wav)/SR:.2f}s | RTF={elapsed / (len(wav)/SR):.3f}')

out_path = os.path.join(OUT_DIR, 'bimpe_ft_generated.wav')
save_wav(out_path, wav)
display(ipd.Audio(wav, rate=SR))


### Stronger voice clone (more reference style)
Lower `alpha` / `beta` → closer to the reference speaker.

In [ ]:
wav2 = inference(
    text, ref_s,
    alpha=0.1, beta=0.5,
    diffusion_steps=10,
    embedding_scale=1,
    speed=1.0,
)
save_wav(os.path.join(OUT_DIR, 'bimpe_ft_strong_ref.wav'), wav2)
display(ipd.Audio(wav2, rate=SR))


### Quick custom line

In [ ]:
custom = 'Let your own intuition be your guide.'
wav3 = inference(custom, ref_s, alpha=0.3, beta=0.7, diffusion_steps=10, speed=1.0)
print(custom)
save_wav(os.path.join(OUT_DIR, 'bimpe_ft_custom.wav'), wav3)
display(ipd.Audio(wav3, rate=SR))


## Troubleshooting

| Problem | Fix |
|---|---|
| `DTensor` import error | `pip install transformers==4.51.3` (torch 2.4.x) |
| `mask_from_lens` import error | Install Resemble fork or rely on `utils.py` fallback |
| `_center` / numpy error | `pip install numpy==1.26.4` |
| Missing checkpoint | Run finetune, or set `CHECKPOINT_PATH` |
| Token length / ALBERT 512 error | Sentence too long — add periods |
| Robotic / wrong voice | Better `REF_WAV`; try `alpha=0.1, beta=0.5` |
| Speech too slow/fast | Set `speed` (`>1` faster, `<1` slower); scales `pred_dur` |
| Slow generation / high RTF | Lower `diffusion_steps` (e.g. 5–8); generation time, not speaking rate |
| phonemizer / espeak errors | Install `espeak-ng` system package |
